# Práctica · Semana 6: Aprendizaje automático y clasificador bayesiano ingenuo

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 2 h 30 min
**Relación con el capítulo:** secciones 6.1 a 6.6

En la semana 5 un experto escribía las reglas. Esta semana el programa **aprende de ejemplos**. Vas a programar desde cero el clasificador bayesiano ingenuo, uno de los métodos de aprendizaje más antiguos y todavía muy usados, y a aplicarlo a tres problemas:

| Parte | Problema | Qué practicas |
|---|---|---|
| 1 | Una prueba médica | Probabilidad condicional y teorema de Bayes |
| 2 | ¿Se juega el partido? | Bayes ingenuo con atributos categóricos, calculado paso a paso |
| 3 y 4 | Filtro de *spam* en español | Bolsa de palabras, suavizado de Laplace, evaluación con datos de prueba |
| 5 | Identificar el idioma de un texto | Bayes ingenuo con trigramas de caracteres |
| 6 | Agrupar sin etiquetas | Un vistazo al aprendizaje no supervisado |

Ejecuta las celdas en orden. Las celdas marcadas con ✏️ son ejercicios. Todas las celdas se probaron con Python 3 y usan librerías que Google Colab trae instaladas.

In [ ]:
import math
import random
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

## 1. Probabilidad y teorema de Bayes

Una enfermedad afecta al **1 %** de la población. Una prueba la detecta en el **90 %** de los enfermos (sensibilidad), pero también da positivo en el **9 %** de los sanos (falsos positivos). Si una persona da positivo, ¿qué probabilidad hay de que esté enferma?

El **teorema de Bayes** (sección 6.4) responde:

$$
P(E \mid +) = \frac{P(+ \mid E)\, P(E)}{P(+)} = \frac{P(+ \mid E)\, P(E)}{P(+ \mid E)\, P(E) + P(+ \mid \neg E)\, P(\neg E)}
$$

In [ ]:
p_e = 0.01             # P(E): prevalencia (probabilidad a priori)
p_pos_e = 0.90         # P(+ | E): sensibilidad
p_pos_no_e = 0.09      # P(+ | ¬E): tasa de falsos positivos

p_pos = p_pos_e * p_e + p_pos_no_e * (1 - p_e)       # probabilidad total
p_e_pos = p_pos_e * p_e / p_pos                       # teorema de Bayes
print(f"P(+)     = {p_pos:.4f}")
print(f"P(E | +) = {p_e_pos:.4f}  ->  {100 * p_e_pos:.1f} %")

Solo un **9,2 %**. La intuición suele decir «90 %», porque confunde $P(+ \mid E)$ con $P(E \mid +)$. Comprobémoslo simulando 100.000 personas.

In [ ]:
rng = random.Random(1)
positivos = enfermos_positivos = 0
for _ in range(100_000):
    enfermo = rng.random() < p_e
    positivo = rng.random() < (p_pos_e if enfermo else p_pos_no_e)
    if positivo:
        positivos += 1
        enfermos_positivos += enfermo
print(f"De {positivos} positivos, {enfermos_positivos} están enfermos: {100 * enfermos_positivos / positivos:.1f} %")

### ✏️ Ejercicio 1. Repetir la prueba

1. Si la persona se hace una **segunda prueba** independiente y vuelve a dar positivo, ¿cuál es ahora la probabilidad de que esté enferma? Pista: usa 0,092 como nueva probabilidad a priori.
2. ¿Qué pasa con $P(E \mid +)$ si la prevalencia es del 20 % en lugar del 1 %? Explica por qué la misma prueba es más o menos útil según la población en la que se aplica.

In [ ]:
def bayes(prior, verosimilitud, falso_positivo):
    return verosimilitud * prior / (verosimilitud * prior + falso_positivo * (1 - prior))

# Escribe aquí tus cálculos

*Tus respuestas al ejercicio 1:*

## 2. Bayes ingenuo paso a paso: ¿se juega el partido?

Un club registró durante 14 días el clima y si se jugó o no un partido al aire libre. Es un conjunto de datos clásico de la enseñanza del aprendizaje automático (Mitchell, 1997, tabla 3.2), traducido al español.

Queremos predecir la **clase** (`juega`) a partir de cuatro **atributos**. El clasificador bayesiano ingenuo elige la clase $c$ que maximiza

$$
P(c) \prod_i P(x_i \mid c)
$$

suponiendo que los atributos son **independientes entre sí dada la clase** (la suposición «ingenua», sección 6.5).

In [ ]:
ATRIBUTOS = ["cielo", "temperatura", "humedad", "viento"]
DIAS = [
    ("soleado", "calurosa", "alta", "débil", "no"),
    ("soleado", "calurosa", "alta", "fuerte", "no"),
    ("nublado", "calurosa", "alta", "débil", "sí"),
    ("lluvioso", "templada", "alta", "débil", "sí"),
    ("lluvioso", "fresca", "normal", "débil", "sí"),
    ("lluvioso", "fresca", "normal", "fuerte", "no"),
    ("nublado", "fresca", "normal", "fuerte", "sí"),
    ("soleado", "templada", "alta", "débil", "no"),
    ("soleado", "fresca", "normal", "débil", "sí"),
    ("lluvioso", "templada", "normal", "débil", "sí"),
    ("soleado", "templada", "normal", "fuerte", "sí"),
    ("nublado", "templada", "alta", "fuerte", "sí"),
    ("nublado", "calurosa", "normal", "débil", "sí"),
    ("lluvioso", "templada", "alta", "fuerte", "no"),
]
print(f"{len(DIAS)} días; clases: {Counter(d[-1] for d in DIAS)}")

In [ ]:
class BayesIngenuoCategorico:
    # Clasificador bayesiano ingenuo para atributos categóricos, con suavizado de Laplace opcional.

    def __init__(self, alfa=0.0):
        self.alfa = alfa                                   # 0 = sin suavizado; 1 = Laplace

    def entrenar(self, ejemplos):
        self.clases = Counter(e[-1] for e in ejemplos)
        self.n = len(ejemplos)
        self.cuentas = defaultdict(Counter)                # (atributo, clase) -> Counter de valores
        self.valores = defaultdict(set)
        for *x, c in ejemplos:
            for i, v in enumerate(x):
                self.cuentas[(i, c)][v] += 1
                self.valores[i].add(v)
        return self

    def p_clase(self, c):
        return self.clases[c] / self.n

    def p_valor(self, i, v, c):
        k = len(self.valores[i])
        return (self.cuentas[(i, c)][v] + self.alfa) / (self.clases[c] + self.alfa * k)

    def puntajes(self, x):
        return {c: self.p_clase(c) * math.prod(self.p_valor(i, v, c) for i, v in enumerate(x))
                for c in self.clases}

    def predecir(self, x):
        p = self.puntajes(x)
        return max(p, key=p.get)


modelo = BayesIngenuoCategorico().entrenar(DIAS)
x = ("soleado", "fresca", "alta", "fuerte")
print("Día nuevo:", dict(zip(ATRIBUTOS, x)))
for c in ["sí", "no"]:
    factores = [modelo.p_valor(i, v, c) for i, v in enumerate(x)]
    print(f"P({c}) = {modelo.p_clase(c):.3f}  ×  " + "  ×  ".join(
        f"P({v}|{c})={f:.3f}" for v, f in zip(x, factores)) + f"  =  {modelo.puntajes(x)[c]:.4f}")
p = modelo.puntajes(x)
print("Predicción:", modelo.predecir(x), "| probabilidades normalizadas:",
      {c: round(v / sum(p.values()), 3) for c, v in p.items()})

### ✏️ Ejercicio 2. El problema de la probabilidad cero

1. Predice la clase de un día `("nublado", "calurosa", "alta", "fuerte")`. ¿Cuál es el puntaje de la clase «no»? ¿Por qué?
2. En los 14 días, nunca se dejó de jugar con cielo nublado. ¿Es razonable concluir que con cielo nublado **nunca** se deja de jugar? ¿Qué pasaría con un solo día nublado sin partido en los datos de prueba?
3. Repite la predicción con el **suavizado de Laplace** (`alfa=1`). ¿Cambia la predicción? ¿Y las probabilidades?

In [ ]:
x2 = ("nublado", "calurosa", "alta", "fuerte")
print("Sin suavizado:", BayesIngenuoCategorico(alfa=0).entrenar(DIAS).puntajes(x2))
print("Con Laplace:  ", BayesIngenuoCategorico(alfa=1).entrenar(DIAS).puntajes(x2))

*Tus respuestas al ejercicio 2:*

## 3. Un filtro de *spam* en español

Ahora los datos son **no estructurados**: textos. Los mensajes de la celda siguiente se escribieron para esta práctica e imitan mensajes reales que llegan por correo, SMS o WhatsApp en Colombia. Cada uno tiene una etiqueta: `spam` o `ham` (mensaje legítimo).

Para que un clasificador pueda usar un texto, lo convertimos en una **bolsa de palabras**: contamos cuántas veces aparece cada palabra e ignoramos el orden (sección 6.5.3).

In [ ]:
MENSAJES = [
    ("spam", "Felicitaciones ganaste un celular nuevo reclama tu premio ya en este enlace"),
    ("spam", "Tu cuenta bancaria fue bloqueada ingresa aquí para verificar tus datos"),
    ("spam", "Gana dinero desde casa sin experiencia escríbenos al whatsapp"),
    ("spam", "Oferta exclusiva 70 por ciento de descuento solo hoy compra ya"),
    ("spam", "Has sido seleccionado para un crédito aprobado sin papeleo responde SI"),
    ("spam", "Premio garantizado gratis haz clic y reclama tu bono"),
    ("spam", "Urgente actualiza tu clave del banco o tu cuenta será suspendida"),
    ("spam", "Invierte 100 mil y gana un millón en una semana garantizado"),
    ("spam", "Ganaste un viaje gratis a San Andrés llama ya para reclamar"),
    ("spam", "Tu paquete está retenido paga el envío en este enlace"),
    ("spam", "Préstamo inmediato sin fiador ni centrales de riesgo escríbenos"),
    ("spam", "Última oportunidad descuento exclusivo en celulares compra hoy"),
    ("spam", "Felicitaciones tu número ganó la rifa reclama el premio"),
    ("spam", "Verifica tus datos bancarios para evitar el bloqueo de tu tarjeta"),
    ("spam", "Trabajo desde casa gana dinero fácil sin inversión"),
    ("spam", "Bono de regalo gratis para los primeros 100 clientes haz clic"),
    ("spam", "Tu suscripción vence hoy renueva ya con descuento exclusivo"),
    ("spam", "Gana un iPhone respondiendo esta encuesta gratis"),
    ("spam", "Crédito preaprobado de cinco millones responde para recibirlo"),
    ("spam", "Alerta de seguridad ingresa al enlace y confirma tu clave"),
    ("spam", "Oferta solo hoy dos por uno en todos los productos compra ya"),
    ("spam", "Recibe dinero gratis por invitar amigos registrate en el enlace"),
    ("spam", "Has ganado un premio en efectivo comunícate de inmediato"),
    ("spam", "Descuento exclusivo para ti reclama tu cupón gratis"),
    ("spam", "Tu tarjeta fue suspendida verifica tus datos en este enlace"),
    ("spam", "Multiplica tu dinero con criptomonedas ganancia garantizada"),
    ("spam", "Promoción imperdible celulares a mitad de precio compra ya"),
    ("spam", "Felicitaciones fuiste elegido ganador de un bono reclama ahora"),
    ("spam", "Actualiza tus datos o perderás tu cuenta hoy mismo"),
    ("spam", "Gana premios todos los días jugando gratis haz clic"),
    ("spam", "Cupo de crédito aprobado sin requisitos llama ya"),
    ("spam", "Oferta especial descuento en planes de datos ilimitados solo hoy"),
    ("spam", "Recibiste una transferencia confirma tu clave para liberarla"),
    ("spam", "Gana dinero extra vendiendo desde tu celular escríbenos"),
    ("spam", "Premio sorpresa gratis para ti reclama en el enlace"),
    ("spam", "Ganaste un mercado gratis reclama tu bono hoy"),
    ("spam", "Inversión segura gana el doble de tu dinero garantizado"),
    ("spam", "Tu cuenta presenta movimientos sospechosos ingresa al enlace"),
    ("spam", "Descuento exclusivo en televisores compra ya antes de que se agoten"),
    ("spam", "Reclama tu subsidio gratis registrando tus datos en el enlace"),
    ("ham", "Hola nos vemos mañana en la sesión de Teams a las siete"),
    ("ham", "Profe ya subí la actividad evaluativa al repositorio"),
    ("ham", "Recuerda que la reunión del equipo es el jueves a las tres"),
    ("ham", "Mamá llego tarde hoy porque tengo clase en la noche"),
    ("ham", "Te comparto el enlace del repositorio para revisar el código"),
    ("ham", "Ya terminé el informe te lo envío para que lo revises"),
    ("ham", "Gracias por la ayuda con el cuaderno de Colab funcionó"),
    ("ham", "El parcial de bases de datos se movió para el lunes"),
    ("ham", "Vamos a almorzar a la una en la cafetería"),
    ("ham", "Mañana no hay clase la profesora está enferma"),
    ("ham", "Por favor revisa el pull request antes de las cinco"),
    ("ham", "Hola cómo estás hace rato no hablamos"),
    ("ham", "La factura del servicio de internet llegó por correo"),
    ("ham", "Te llamo en la noche para cuadrar lo del viaje"),
    ("ham", "Ya pagué el arriendo de este mes"),
    ("ham", "El equipo de soporte reinició el servidor de pruebas"),
    ("ham", "Recuerda traer el portátil para la práctica del sábado"),
    ("ham", "Feliz cumpleaños que tengas un gran día"),
    ("ham", "Necesito que me envíes la presentación antes de la reunión"),
    ("ham", "El despliegue a producción quedó para el viernes"),
    ("ham", "Compré el mercado para la semana ya está en la nevera"),
    ("ham", "La tarea de inteligencia artificial es para el domingo"),
    ("ham", "Hola profe tengo una duda sobre el algoritmo A estrella"),
    ("ham", "Te espero en la entrada de la universidad a las seis"),
    ("ham", "Mi papá pregunta si vienes a cenar el domingo"),
    ("ham", "Ya corregí el error del formulario de registro"),
    ("ham", "El banco me confirmó que la transferencia llegó bien"),
    ("ham", "Hoy hay partido de fútbol a las ocho nos vemos en la cancha"),
    ("ham", "Me puedes prestar los apuntes de la clase de ayer"),
    ("ham", "La reunión con el cliente se pasó para la próxima semana"),
    ("ham", "Gracias por el regalo me encantó"),
    ("ham", "Ya actualicé la documentación del proyecto en el repositorio"),
    ("ham", "Avísame cuando llegues a la casa"),
    ("ham", "El curso de Python empieza el próximo mes"),
    ("ham", "Te mando la foto del tablero con los ejercicios"),
    ("ham", "Mañana tengo cita médica a las nueve llego después"),
    ("ham", "Los resultados de las pruebas unitarias salieron bien"),
    ("ham", "Podemos hacer la videollamada a las cuatro"),
    ("ham", "El grupo de estudio se reúne en la biblioteca"),
    ("ham", "Ya está lista la base de datos para la demo"),
]
print(Counter(etiqueta for etiqueta, _ in MENSAJES))

In [ ]:
def tokenizar(texto):
    # Minúsculas y palabras de letras o números. Es la tokenización más simple posible.
    return re.findall(r"[a-záéíóúñü0-9]+", texto.lower())


print(tokenizar("¡Felicitaciones! Ganaste un celular: reclama tu PREMIO ya."))

### 3.1 Bayes ingenuo multinomial

Con textos se usa la variante **multinomial**: la probabilidad de un mensaje dada la clase es el producto de las probabilidades de cada palabra dada la clase. Dos detalles prácticos (sección 6.5.4):

- **Suavizado de Laplace:** se suma 1 a todas las cuentas para que una palabra nunca vista en una clase no anule el producto.
- **Logaritmos:** el producto de cientos de probabilidades pequeñas se vuelve 0 en el computador. Por eso se suman los **logaritmos** de las probabilidades.

In [ ]:
class BayesIngenuoTexto:
    def __init__(self, alfa=1.0, tokenizador=tokenizar):
        self.alfa, self.tokenizar = alfa, tokenizador

    def entrenar(self, textos, etiquetas):
        self.clases = Counter(etiquetas)
        self.palabras = {c: Counter() for c in self.clases}
        for t, c in zip(textos, etiquetas):
            self.palabras[c].update(self.tokenizar(t))
        self.vocabulario = set().union(*self.palabras.values())
        self.totales = {c: sum(self.palabras[c].values()) for c in self.clases}
        return self

    def log_p_palabra(self, w, c):
        V = len(self.vocabulario)
        return math.log((self.palabras[c][w] + self.alfa) / (self.totales[c] + self.alfa * V))

    def log_puntajes(self, texto):
        n = sum(self.clases.values())
        tokens = [w for w in self.tokenizar(texto) if w in self.vocabulario]
        return {c: math.log(self.clases[c] / n) + sum(self.log_p_palabra(w, c) for w in tokens)
                for c in self.clases}

    def predecir(self, texto):
        p = self.log_puntajes(texto)
        return max(p, key=p.get)

    def probabilidades(self, texto):
        lp = self.log_puntajes(texto)
        m = max(lp.values())
        exp = {c: math.exp(v - m) for c, v in lp.items()}
        s = sum(exp.values())
        return {c: v / s for c, v in exp.items()}

### 3.2 Entrenar y evaluar

Para saber si el filtro **generaliza**, hay que evaluarlo con mensajes que **no vio** durante el entrenamiento. Separamos los datos al azar: 75 % para entrenar y 25 % para probar (sección 6.2.4).

In [ ]:
def dividir(datos, prop_prueba=0.25, semilla=0):
    datos = list(datos)
    random.Random(semilla).shuffle(datos)
    corte = int(len(datos) * (1 - prop_prueba))
    return datos[:corte], datos[corte:]


entrenamiento, prueba = dividir(MENSAJES)
filtro = BayesIngenuoTexto().entrenar([t for _, t in entrenamiento], [c for c, _ in entrenamiento])


def evaluar(modelo, datos, positiva="spam"):
    reales = [c for c, _ in datos]
    predichas = [modelo.predecir(t) for _, t in datos]
    vp = sum(r == positiva and p == positiva for r, p in zip(reales, predichas))
    fp = sum(r != positiva and p == positiva for r, p in zip(reales, predichas))
    fn = sum(r == positiva and p != positiva for r, p in zip(reales, predichas))
    vn = sum(r != positiva and p != positiva for r, p in zip(reales, predichas))
    exactitud = (vp + vn) / len(datos)
    precision = vp / (vp + fp) if vp + fp else 0.0
    exhaustividad = vp / (vp + fn) if vp + fn else 0.0
    return {"VP": vp, "FP": fp, "FN": fn, "VN": vn, "exactitud": round(exactitud, 3),
            "precisión": round(precision, 3), "exhaustividad": round(exhaustividad, 3)}


print(f"Entrenamiento: {len(entrenamiento)} mensajes | prueba: {len(prueba)} mensajes")
print("Resultados en los datos de prueba:", evaluar(filtro, prueba))
errores = [(c, t) for c, t in prueba if filtro.predecir(t) != c]
print("\nErrores:", "ninguno" if not errores else "")
for c, t in errores:
    print(f"  real={c:<5} predicho={filtro.predecir(t):<5} | {t}")

La **matriz de confusión** (VP, FP, FN, VN) separa los dos tipos de error, que en un filtro de *spam* no cuestan lo mismo (sección 6.2.4):

| | Predicho *spam* | Predicho *ham* |
|---|:-:|:-:|
| **Realmente *spam*** | VP (verdadero positivo) | FN (*spam* que se cuela) |
| **Realmente *ham*** | FP (mensaje legítimo perdido) | VN (verdadero negativo) |

- **Precisión:** de lo que el filtro marcó como *spam*, ¿cuánto lo era? $\text{VP} / (\text{VP} + \text{FP})$.
- **Exhaustividad** (*recall*): del *spam* real, ¿cuánto detectó? $\text{VP} / (\text{VP} + \text{FN})$.

### 3.3 ¿Qué aprendió el filtro?

Una ventaja de Bayes ingenuo es que se puede inspeccionar. Las palabras con mayor cociente $P(w \mid \text{spam}) / P(w \mid \text{ham})$ son las que más «delatan» el *spam*.

In [ ]:
def palabras_indicativas(modelo, clase, otra, k=12):
    puntaje = {w: modelo.log_p_palabra(w, clase) - modelo.log_p_palabra(w, otra) for w in modelo.vocabulario}
    return sorted(puntaje, key=puntaje.get, reverse=True)[:k]


print("Más indicativas de spam:", palabras_indicativas(filtro, "spam", "ham"))
print("Más indicativas de ham: ", palabras_indicativas(filtro, "ham", "spam"))
for texto in ["Hola profe adjunto el enlace del repositorio con el informe",
              "Reclama gratis tu premio en el enlace",
              "El banco confirmó tu transferencia ingresa tu clave para verla"]:
    print(f"\n«{texto}»\n  -> {filtro.predecir(texto)}  {({c: round(p, 3) for c, p in filtro.probabilidades(texto).items()})}")

### ✏️ Ejercicio 3. Poner a prueba el filtro

1. Escribe **cinco mensajes** propios, algunos legítimos y otros *spam*, incluyendo al menos uno diseñado para **engañar** al filtro (por ejemplo, un *spam* escrito con palabras de mensajes legítimos). ¿Cuántos clasifica bien?
2. Con 80 mensajes, la evaluación depende mucho de qué mensajes caen en la prueba. Ejecuta la celda de **validación con varias divisiones** y observa la variación de la exactitud. ¿Qué te dice eso sobre los resultados obtenidos con conjuntos de datos pequeños?
3. En un filtro de *spam*, ¿qué error es más grave: un falso positivo o un falso negativo? ¿Cómo podrías modificar la regla de decisión para cometer menos del error que consideras más grave? Pista: en lugar de elegir la clase más probable, marca *spam* solo si $P(\text{spam} \mid \text{mensaje}) > 0{,}9$.

In [ ]:
mis_mensajes = [
    "Escribe aquí tu primer mensaje",
]
for m in mis_mensajes:
    print(filtro.predecir(m), "|", m)

In [ ]:
exactitudes = []
for semilla in range(20):
    tr, te = dividir(MENSAJES, semilla=semilla)
    f = BayesIngenuoTexto().entrenar([t for _, t in tr], [c for c, _ in tr])
    exactitudes.append(evaluar(f, te)["exactitud"])
print("Exactitud en 20 divisiones distintas:", exactitudes)
print(f"Media: {np.mean(exactitudes):.3f} | desviación estándar: {np.std(exactitudes):.3f} | "
      f"mínimo: {min(exactitudes):.3f} | máximo: {max(exactitudes):.3f}")

*Tus respuestas al ejercicio 3:*

## 4. Con datos reales: el corpus *SMS Spam Collection*

Ochenta mensajes son muy pocos. El corpus *SMS Spam Collection* (Almeida y Hidalgo, 2011) reúne **5.574 SMS reales en inglés**, etiquetados como *spam* o *ham*. Está publicado en el repositorio de la Universidad de California en Irvine (UCI) con licencia CC BY 4.0.

La celda lo descarga y entrena el mismo clasificador. Si no tienes conexión, la celda te lo indica y puedes continuar.

In [ ]:
import io
import urllib.request
import zipfile

URL_SMS = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
try:
    datos_zip = urllib.request.urlopen(URL_SMS, timeout=30).read()
    lineas = zipfile.ZipFile(io.BytesIO(datos_zip)).read("SMSSpamCollection").decode("utf-8", "ignore").splitlines()
    SMS = [tuple(l.split("\t", 1)) for l in lineas if "\t" in l]
    print(f"{len(SMS)} mensajes:", Counter(c for c, _ in SMS))
except Exception as e:
    SMS = []
    print("No se pudo descargar el corpus:", e)

In [ ]:
if SMS:
    tr, te = dividir(SMS, prop_prueba=0.25, semilla=0)
    filtro_sms = BayesIngenuoTexto().entrenar([t for _, t in tr], [c for c, _ in tr])
    print("Resultados en la prueba:", evaluar(filtro_sms, te))
    print("Más indicativas de spam:", palabras_indicativas(filtro_sms, "spam", "ham"))

    # Comparación con la implementación de scikit-learn
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics import accuracy_score
    vec = CountVectorizer(token_pattern=r"[a-záéíóúñü0-9]+", lowercase=True)
    Xtr = vec.fit_transform([t for _, t in tr])
    nb = MultinomialNB(alpha=1.0).fit(Xtr, [c for c, _ in tr])
    pred = nb.predict(vec.transform([t for _, t in te]))
    print("Exactitud de scikit-learn:", round(accuracy_score([c for c, _ in te], pred), 3))

### ✏️ Ejercicio 4. Datos desbalanceados

1. En el corpus, solo el 13 % de los mensajes son *spam*. Un clasificador que **siempre** dijera *ham* tendría una exactitud del 87 %. ¿Por qué la exactitud sola no basta para evaluar un filtro? ¿Qué métricas mirarías?
2. ¿Tu implementación y la de scikit-learn dan una exactitud parecida? ¿Por qué no tendría sentido usar el filtro entrenado con SMS en inglés para los mensajes en español de la parte 3?

*Tus respuestas al ejercicio 4:*

## 5. Identificar el idioma de un texto

El mismo algoritmo sirve para otro problema si cambiamos la representación. Para identificar el idioma, en lugar de palabras usamos **trigramas de caracteres**: secuencias de tres letras como `ció`, `the` o `ção`, que son muy características de cada idioma.

In [ ]:
def trigramas(texto):
    t = " " + re.sub(r"[^a-záéíóúñüàâçèêëîïôœùûãõ ]", "", texto.lower()) + " "
    return [t[i:i + 3] for i in range(len(t) - 2)]


print(trigramas("La inteligencia"))

FRASES = {
    "español": [
        "el aprendizaje automático permite que los programas mejoren con la experiencia",
        "mañana tenemos clase de inteligencia artificial en la universidad",
        "los datos de entrenamiento deben representar bien el problema",
        "la búsqueda en amplitud explora el árbol por niveles",
        "hoy hace mucho calor en la ciudad y no hay viento",
        "la estudiante publicó su proyecto en un repositorio público",
        "el clasificador bayesiano ingenuo es rápido y fácil de entender",
        "necesitamos una función de evaluación para el juego",
    ],
    "inglés": [
        "machine learning allows programs to improve with experience",
        "tomorrow we have an artificial intelligence class at the university",
        "the training data must represent the problem well",
        "breadth first search explores the tree level by level",
        "today it is very hot in the city and there is no wind",
        "the student published her project in a public repository",
        "the naive bayes classifier is fast and easy to understand",
        "we need an evaluation function for the game",
    ],
    "portugués": [
        "a aprendizagem automática permite que os programas melhorem com a experiência",
        "amanhã temos aula de inteligência artificial na universidade",
        "os dados de treinamento devem representar bem o problema",
        "a busca em largura explora a árvore por níveis",
        "hoje faz muito calor na cidade e não há vento",
        "a estudante publicou o seu projeto num repositório público",
        "o classificador bayesiano ingénuo é rápido e fácil de entender",
        "precisamos de uma função de avaliação para o jogo",
    ],
    "francés": [
        "l'apprentissage automatique permet aux programmes de s'améliorer avec l'expérience",
        "demain nous avons un cours d'intelligence artificielle à l'université",
        "les données d'entraînement doivent bien représenter le problème",
        "la recherche en largeur explore l'arbre niveau par niveau",
        "aujourd'hui il fait très chaud dans la ville et il n'y a pas de vent",
        "l'étudiante a publié son projet dans un dépôt public",
        "le classifieur bayésien naïf est rapide et facile à comprendre",
        "nous avons besoin d'une fonction d'évaluation pour le jeu",
    ],
}
textos = [f for frases in FRASES.values() for f in frases]
idiomas = [i for i, frases in FRASES.items() for _ in frases]
detector = BayesIngenuoTexto(alfa=0.5, tokenizador=trigramas).entrenar(textos, idiomas)

for frase in ["¿Dónde queda la biblioteca?", "Where is the library?", "Onde fica a biblioteca?",
              "Où se trouve la bibliothèque ?", "Obrigado pela ajuda", "Merci beaucoup pour tout",
              "Thanks for the help", "Gracias por todo"]:
    print(f"{frase:<32} -> {detector.predecir(frase)}")

### ✏️ Ejercicio 5. ¿Dónde se confunde?

1. Escribe frases cortas que confundan al detector. ¿Entre qué idiomas se equivoca más? ¿Por qué?
2. Añade un quinto idioma (por ejemplo, italiano) con al menos 8 frases y comprueba si lo reconoce.
3. Prueba a usar **palabras** (`tokenizar`) en lugar de trigramas con frases que no aparecen en el entrenamiento. ¿Cuál representación funciona mejor con tan pocos datos? ¿Por qué?

In [ ]:
# Escribe aquí tus pruebas

*Tus respuestas al ejercicio 5:*

## 6. Un vistazo al aprendizaje no supervisado

En todos los ejemplos anteriores los datos tenían **etiquetas**: alguien había dicho qué mensajes eran *spam*. En el **aprendizaje no supervisado** no hay etiquetas: el algoritmo busca por sí mismo **grupos** de datos parecidos (sección 6.2.2).

El algoritmo ***k*-medias** coloca *k* centros, asigna cada punto al centro más cercano, mueve cada centro al promedio de sus puntos y repite. Lo aplicamos a 300 clientes ficticios de una tienda en línea, descritos por su gasto mensual y su número de visitas.

In [ ]:
from sklearn.cluster import KMeans

rng_np = np.random.default_rng(3)
medias = [(50, 2), (300, 6), (150, 15)]          # tres tipos de cliente, desconocidos para el algoritmo
clientes = np.vstack([rng_np.normal(m, (25, 1.5), size=(100, 2)) for m in medias])

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(clientes)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
a1.scatter(clientes[:, 0], clientes[:, 1], s=10, color="gray")
a1.set_title("Datos sin etiquetas")
a2.scatter(clientes[:, 0], clientes[:, 1], s=10, c=km.labels_, cmap="viridis")
a2.scatter(*km.cluster_centers_.T, marker="X", s=200, color="red")
a2.set_title("Grupos encontrados por k-medias (k = 3)")
for a in (a1, a2):
    a.set_xlabel("Gasto mensual (miles de pesos)")
    a.set_ylabel("Visitas al mes")
plt.tight_layout()
plt.show()
print("Centros encontrados:\n", km.cluster_centers_.round(1))

### ✏️ Ejercicio 6. ¿Qué significan los grupos?

1. Describe con palabras cada grupo encontrado, como lo haría un equipo de mercadeo.
2. Cambia `n_clusters` a 2 y a 5. ¿Cómo sabrías cuál es el número «correcto» de grupos si no conocieras los datos de antemano?
3. ¿En qué se diferencia este problema del filtro de *spam*? ¿Qué tipo de aprendizaje usaría una aplicación de música para recomendarte canciones? ¿Y un programa que aprende a jugar Conecta 4 jugando contra sí mismo?

*Tus respuestas al ejercicio 6:*

## 7. Cierre

Completa esta tabla con lo que observaste:

| Concepto | Qué significa | Dónde lo viste en la práctica |
|---|---|---|
| Probabilidad a priori | | |
| Verosimilitud | | |
| Suposición ingenua | | |
| Suavizado de Laplace | | |
| Datos de entrenamiento y de prueba | | |
| Precisión y exhaustividad | | |
| Aprendizaje no supervisado | | |

Lleva el cuaderno ejecutado a la sesión sincrónica. El clasificador `BayesIngenuoCategorico` es la base de la parte B de la **Actividad Evaluativa 3**, que se entrega esta semana.

---

**Referencias**

Almeida, T. A., y Hidalgo, J. M. G. (2011). *SMS Spam Collection* [Conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84

Mitchell, T. M. (1997). *Machine learning*. McGraw-Hill.

Russell, S. J., y Norvig, P. (2004). *Inteligencia artificial: Un enfoque moderno* (2.ª ed.; J. M. Corchado Rodríguez et al., Trads.). Pearson Educación.